<a href="https://colab.research.google.com/github/stevenolanecon/7002LBSAI/blob/main/notebooks/week2_instructor_ci_coverage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 – Instructor demo: 95% CI coverage, re-drawable live

**For live use in the workshop, not a student handout.** The slide's own "100 confidence intervals" chart is baked in at a fixed draw (a specific random seed) – always the same picture. This notebook draws 100 fresh samples, builds a 95% CI for each, and colours each one by whether it actually contains the true population mean.

**Why re-draw live:** a single static image invites students to think 95% coverage is a guarantee about *that* image. Change the seed below and re-run – the count out of 100 will land somewhere in the mid-90s each time, sometimes above 95, sometimes below, never exactly 95. That variability *is* the point: 95% is a long-run average property of the procedure, not a promise about any one batch of 100.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Same population as the Week 1/2 reaction-time simulation
np.random.seed(42)
n_students = 15
true_means = np.clip(np.random.normal(280, 50, n_students), 180, 420)
within_sd = 40.0
mu_pop = float(true_means.mean())
sd_pop = float(np.sqrt(np.var(true_means) + within_sd ** 2))
print(f'True population mean: {mu_pop:.1f} ms')

## Draw 100 samples and build a 95% CI for each

**Change `SEED` and re-run this cell for a fresh draw live in class.**

In [ ]:
SEED = 77  # <-- change this and re-run for a fresh draw

np.random.seed(SEED)
n_draws = 100
rows = []
for i in range(n_draws):
    sample = np.random.normal(mu_pop, sd_pop, n_students)
    m = sample.mean()
    se = sample.std(ddof=1) / np.sqrt(n_students)
    lo, hi = m - 1.96 * se, m + 1.96 * se
    covers = lo <= mu_pop <= hi
    rows.append((i, m, lo, hi, covers))

n_covers = sum(r[4] for r in rows)
print(f'{n_covers} / {n_draws} intervals contain the true mean ({mu_pop:.1f} ms)')

## Visualise it

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))
for i, m, lo, hi, covers in rows:
    colour = '#2d6a4f' if covers else '#e63946'
    ax.plot([lo, hi], [i, i], color=colour, linewidth=1.5)
    ax.plot(m, i, 'o', color=colour, markersize=3)

ax.axvline(mu_pop, color='#1a1a2e', linestyle='--', linewidth=1.5, label=f'True mean ({mu_pop:.1f} ms)')
ax.set_xlabel('Reaction time (ms)')
ax.set_ylabel('Sample #')
ax.set_title(f'{n_covers} / {n_draws} of the 95% CIs contain the true mean')
ax.legend()
plt.tight_layout()
plt.show()

## Live talking points

- Re-run the two cells above with a different `SEED` a couple of times in front of the class. The count will move around the mid-90s – ask the class to call out the new number each time.
- Ask: "does a 95% CI mean there's a 95% chance the true mean is in *this* interval?" – no. The true mean is fixed; it's the interval that's random from sample to sample. 95% describes the procedure across many hypothetical repeats, not any single interval.
- If someone asks "what if we used 99% instead?" – change `1.96` to `2.576` in the loop above and re-run; the count should climb toward 99, and the intervals will visibly widen.